# Task 4 – Benchmark con scikit-learn
Aplicamos modelos de `sklearn` y comparamos sus métricas (accuracy, precision, recall) contra las implementaciones manuales de la tarea previa.

## 1. Preparación de datos
Reutilizamos el mismo conjunto y pipeline de Task 3: eliminamos `url`, codificamos `status`, escogemos las dos características más correlacionadas (`google_index`, `page_rank`) y normalizamos con estadísticas del set de entrenamiento para garantizar comparabilidad.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

RANDOM_SEED = 42

# Carga y limpieza
df = pd.read_csv('dataset_phishing.csv').drop(columns=['url'])
df['status'] = df['status'].map({'legitimate': 0, 'phishing': 1})

# Selección de features más correlacionadas con la etiqueta
corr = df.corr(numeric_only=True)['status'].abs().sort_values(ascending=False)
features = corr.index[1:3].tolist()

X = df[features].to_numpy(dtype=np.float64)
y = df['status'].to_numpy(dtype=np.int64)


def train_test_split_manual(X, y, test_size=0.2, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(X.shape[0])
    rng.shuffle(idx)
    test_count = int(len(idx) * test_size)
    test_idx = idx[:test_count]
    train_idx = idx[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train_raw, X_test_raw, y_train, y_test = train_test_split_manual(X, y, test_size=0.2)
mean_train = X_train_raw.mean(axis=0)
std_train = np.where(X_train_raw.std(axis=0) == 0, 1.0, X_train_raw.std(axis=0))
X_train = (X_train_raw - mean_train) / std_train
X_test = (X_test_raw - mean_train) / std_train

print('Features usadas:', features)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)


Features usadas: ['google_index', 'page_rank']
X_train: (9144, 2) | X_test: (2286, 2)


## 2. Implementaciones manuales con referencia en el task3
Recreamos las funciones sigmoide, costo y descenso de gradiente junto con el KNN basado en distancia euclidiana para obtener las métricas base del ejercicio anterior.

In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def predict_proba_manual(X, w, b):
    return sigmoid(X @ w + b)


def log_loss(y_true, y_prob, eps=1e-9):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))


def compute_gradients(X, y, w, b):
    y_hat = predict_proba_manual(X, w, b)
    error = y_hat - y
    grad_w = (X.T @ error) / X.shape[0]
    grad_b = error.mean()
    return grad_w, grad_b


def gradient_descent(X, y, lr=0.1, epochs=4000):
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(epochs):
        grad_w, grad_b = compute_gradients(X, y, w, b)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b


def predict_labels_manual(X, w, b, threshold=0.5):
    return (predict_proba_manual(X, w, b) >= threshold).astype(int)


def predict_knn_manual(X_train, y_train, X_test, k=3):
    preds = np.empty(X_test.shape[0], dtype=int)
    for i, x in enumerate(X_test):
        distances = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
        nearest = np.argpartition(distances, k)[:k]
        preds[i] = int(y_train[nearest].mean() >= 0.5)
    return preds


In [3]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0)
    }

w_manual, b_manual = gradient_descent(X_train, y_train)
manual_lr_preds = predict_labels_manual(X_test, w_manual, b_manual)
manual_lr_metrics = compute_metrics(y_test, manual_lr_preds)

manual_knn_preds = predict_knn_manual(X_train, y_train, X_test, k=3)
manual_knn_metrics = compute_metrics(y_test, manual_knn_preds)

manual_lr_metrics, manual_knn_metrics

({'accuracy': 0.8648293963254593,
  'precision': 0.8761987794245859,
  'recall': 0.8575085324232082},
 {'accuracy': 0.8490813648293963,
  'precision': 0.8868101028999065,
  'recall': 0.8088737201365188})

## 3. Modelos sklearn
Entrenamos `LogisticRegression` y `KNeighborsClassifier` (k=3) usando los mismos datos normalizados para lograr una comparación justa.

In [4]:
sk_lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
sk_lr.fit(X_train, y_train)
sk_lr_preds = sk_lr.predict(X_test)
sk_lr_metrics = compute_metrics(y_test, sk_lr_preds)

sk_knn = KNeighborsClassifier(n_neighbors=3)
sk_knn.fit(X_train, y_train)
sk_knn_preds = sk_knn.predict(X_test)
sk_knn_metrics = compute_metrics(y_test, sk_knn_preds)

sk_lr_metrics, sk_knn_metrics

({'accuracy': 0.8648293963254593,
  'precision': 0.8761987794245859,
  'recall': 0.8575085324232082},
 {'accuracy': 0.8902012248468941,
  'precision': 0.8692862870890137,
  'recall': 0.9249146757679181})

## 4. Comparativa de métricas
Apilamos las métricas en un `DataFrame` para contrastar rápidamente implementaciones manuales vs versiones de `sklearn`.

In [5]:
comparison = pd.DataFrame([
    {'modelo': 'Manual Logistic', **manual_lr_metrics},
    {'modelo': 'Manual KNN (k=3)', **manual_knn_metrics},
    {'modelo': 'sklearn LogisticRegression', **sk_lr_metrics},
    {'modelo': 'sklearn KNN (k=3)', **sk_knn_metrics},
]).set_index('modelo')

comparison.round(4)

,accuracy,precision,recall
modelo,,,
Manual Logistic,0.8648,0.8762,0.8575
Manual KNN (k=3),0.8491,0.8868,0.8089
sklearn LogisticRegression,0.8648,0.8762,0.8575
sklearn KNN (k=3),0.8902,0.8693,0.9249


## 5. Análisis de cierre
- **¿Qué implementación fue mejor?** Generalmente `sklearn` ofreció ligeras mejoras gracias a optimizaciones numéricas y regularización interna; también evita errores de redondeo o hiperparámetros no ajustados ya por ejemplo la tasa de aprendizaje fija. Esto es debido a que las librerías como `sklearn` están optimizadas y probadas en una amplia variedad de escenarios, lo que les permite manejar mejor las complejidades numéricas y ofrecer un rendimiento más robusto en comparación con implementaciones manuales que pueden ser más susceptibles a errores o ineficiencias.
- **Costo de los errores:** En phishing, un falso negativo porque hicimos que nos dejara pasar un ataque y estoes más grave porque expone credenciales o recursos. Por lo tanto, tomamos la decisión que sí conviene priorizar **recall** ya por su sensibilidad para la clase phishing, aun si se sacrifica algo de precision y se bloquean algunos usuarios legítimos tenemos mejores resultados a largo plazo. 
